# BlinkIT Grocery Sales Analysis
**Student project | Sapphire IQ brief | Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook**

## Business context
This analysis follows the supplied Sapphire IQ project brief. It examines product mix, outlet types and locations, overall recorded sales, ratings, data reliability, sales drivers, comparisons, and practical recommendations.

### Important interpretation note
The workbook contains 8,523 item/outlet records and a `Sales` field, but no transaction date, quantity, profit, order status, or customer-level identifier. Therefore, this project reports **recorded sales** from the supplied `Sales` column. It does not claim to measure time-based growth, profit, order cancellations, or causal effects.


In [ ]:
# If needed, install dependencies in your notebook environment:
# %pip install pandas numpy matplotlib seaborn openpyxl jupyter

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

PROJECT_DIR = Path.cwd()
# If you opened this notebook from the project root, this is the expected path.
DATA_PATH = PROJECT_DIR / "data" / "blinkit_grocery_raw.csv"
if not DATA_PATH.exists():
    # Fallback for running from a notebooks/ subfolder
    DATA_PATH = PROJECT_DIR.parent / "data" / "blinkit_grocery_raw.csv"

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


## Objective 1 — Understand the business and sales performance
**Tasks 1.1–1.4:** understand products, outlets and locations, overall sales, customer ratings, and product performance.


In [ ]:
# Data dictionary and basic profile
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data type": [str(df[c].dtype) for c in df.columns],
    "Non-null values": [df[c].notna().sum() for c in df.columns],
    "Missing values": [df[c].isna().sum() for c in df.columns],
    "Unique values": [df[c].nunique(dropna=True) for c in df.columns],
})
display(data_dictionary)

kpis = pd.DataFrame({
    "Metric": ["Recorded sales total", "Number of records", "Distinct item identifiers",
               "Distinct outlets", "Average rating", "Median rating"],
    "Value": [df["Sales"].sum(), len(df), df["Item Identifier"].nunique(),
              df["Outlet Identifier"].nunique(), df["Rating"].mean(), df["Rating"].median()]
})
display(kpis)


In [ ]:
# Tasks 1.1 and 1.4: product mix and category performance
category_summary = (df.groupby("Item Type", as_index=False)
                    .agg(Record_Count=("Sales", "size"),
                         Total_Sales=("Sales", "sum"),
                         Average_Sales_Per_Record=("Sales", "mean"),
                         Average_Rating=("Rating", "mean"))
                    .sort_values("Total_Sales", ascending=False))
display(category_summary)

plt.figure(figsize=(11, 7))
sns.barplot(data=category_summary, y="Item Type", x="Total_Sales", color="#3b5bdb")
plt.title("Recorded Sales by Item Type")
plt.xlabel("Recorded sales")
plt.ylabel("Item type")
plt.tight_layout()
plt.show()

fat_summary = (df.groupby("Item Fat Content", as_index=False)
               .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                    Average_Sales=("Sales", "mean"))
               .sort_values("Total_Sales", ascending=False))
display(fat_summary)


In [ ]:
# Task 1.2: outlet types, locations and sizes
outlet_type_summary = (df.groupby("Outlet Type", as_index=False)
                       .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                            Average_Sales_Per_Record=("Sales", "mean"),
                            Average_Rating=("Rating", "mean"))
                       .sort_values("Total_Sales", ascending=False))
location_summary = (df.groupby("Outlet Location Type", as_index=False)
                    .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                         Average_Sales_Per_Record=("Sales", "mean"))
                    .sort_values("Total_Sales", ascending=False))
size_summary = (df.groupby("Outlet Size", as_index=False)
                .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                     Average_Sales_Per_Record=("Sales", "mean"))
                .sort_values("Total_Sales", ascending=False))
display(outlet_type_summary)
display(location_summary)
display(size_summary)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.barplot(data=outlet_type_summary, y="Outlet Type", x="Total_Sales", ax=axes[0], color="#4263eb")
axes[0].set_title("Sales by Outlet Type")
sns.barplot(data=location_summary, x="Outlet Location Type", y="Total_Sales", ax=axes[1], color="#15aabf")
axes[1].set_title("Sales by Location Tier")
sns.barplot(data=size_summary, x="Outlet Size", y="Total_Sales", ax=axes[2], color="#7950f2")
axes[2].set_title("Sales by Outlet Size")
for ax in axes:
    ax.set_xlabel("Recorded sales")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## Objective 2 — Make sure the available information is reliable
**Tasks 2.1–2.4:** identify missing values, duplicates, unusually high/low values, and inconsistent labels. The raw data is kept unchanged; cleaning is performed on a separate copy.


In [ ]:
# Task 2.1: missing information
missing_report = pd.DataFrame({
    "Missing_Count": df_raw.isna().sum(),
    "Missing_Percent": (df_raw.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)
display(missing_report[missing_report["Missing_Count"] > 0])

# Task 2.2: duplicate records
duplicate_count = int(df_raw.duplicated().sum())
print(f"Exact duplicate rows: {duplicate_count:,}")
print("Repeated item identifiers are not automatically duplicates: the same item can appear in multiple outlets/records.")


In [ ]:
# Task 2.3: IQR-based screening for unusual numeric values (flags, not automatic errors)
numeric_cols = ["Item Visibility", "Item Weight", "Sales", "Rating"]
outlier_rows = []
for col in numeric_cols:
    s = df_raw[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = ((df_raw[col] < low) | (df_raw[col] > high)).sum()
    outlier_rows.append([col, q1, q3, low, high, int(flagged)])
outlier_report = pd.DataFrame(outlier_rows, columns=[
    "Column", "Q1", "Q3", "IQR_Lower_Bound", "IQR_Upper_Bound", "Flagged_Rows"
])
display(outlier_report)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Item Visibility", "Item Weight", "Sales"]):
    sns.boxplot(y=df_raw[col], ax=ax, color="#748ffc")
    ax.set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2.4: standardize fat-content labels and handle missing item weights
df_clean = df_raw.copy()
df_clean["Item Fat Content"] = df_clean["Item Fat Content"].replace({
    "LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"
})
missing_weight_before = int(df_clean["Item Weight"].isna().sum())

# Impute using the median for the same item identifier, then the item type median.
df_clean["Item Weight"] = df_clean["Item Weight"].fillna(
    df_clean.groupby("Item Identifier")["Item Weight"].transform("median")
)
df_clean["Item Weight"] = df_clean["Item Weight"].fillna(
    df_clean.groupby("Item Type")["Item Weight"].transform("median")
)

print("Fat-content labels after standardization:")
display(df_clean["Item Fat Content"].value_counts().rename_axis("Label").reset_index(name="Count"))
print(f"Missing item weights before: {missing_weight_before:,}")
print(f"Missing item weights after: {df_clean['Item Weight'].isna().sum():,}")
print(f"Exact duplicates in raw data: {df_raw.duplicated().sum():,}")
print("Cleaning note: median imputation preserves rows but introduces estimated weights; use the raw column for analyses where this assumption is not appropriate.")


## Objective 3 — Find out what is driving better sales performance
**Tasks 3.1–3.4:** compare product types and outlets, and examine associations between sales and item visibility, weight, ratings, and outlet characteristics. These are descriptive associations, not proof of causation.


In [ ]:
# Task 3.1: item types and high/low performers
display(category_summary.head(5))
display(category_summary.tail(5).sort_values("Total_Sales"))

# Top individual item identifiers by total recorded sales across their records
item_performance = (df.groupby("Item Identifier", as_index=False)
                    .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                         Average_Sales=("Sales", "mean"), Average_Rating=("Rating", "mean"))
                    .sort_values("Total_Sales", ascending=False))
display(item_performance.head(10))


In [ ]:
# Task 3.2 and 3.4: outlet comparison with both totals and per-record averages
outlet_comparison = (df.groupby(["Outlet Type", "Outlet Identifier"], as_index=False)
                     .agg(Record_Count=("Sales", "size"), Total_Sales=("Sales", "sum"),
                          Average_Sales_Per_Record=("Sales", "mean"),
                          Average_Rating=("Rating", "mean"))
                     .sort_values("Total_Sales", ascending=False))
display(outlet_comparison)

# Outlet establishment year is a cohort attribute, not a transaction timeline.
establishment_summary = (df.groupby("Outlet Establishment Year", as_index=False)
                         .agg(Outlet_Count=("Outlet Identifier", "nunique"),
                              Record_Count=("Sales", "size"),
                              Total_Sales=("Sales", "sum"),
                              Average_Sales_Per_Record=("Sales", "mean"))
                         .sort_values("Outlet Establishment Year"))
display(establishment_summary)


In [ ]:
# Task 3.3: numeric associations
corr_cols = ["Item Visibility", "Item Weight", "Sales", "Rating"]
correlation = df[corr_cols].corr(numeric_only=True)
display(correlation)

plt.figure(figsize=(7, 5))
sns.heatmap(correlation, annot=True, fmt=".3f", cmap="vlag", center=0, square=True)
plt.title("Correlation Between Numeric Fields")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.scatterplot(data=df, x="Item Visibility", y="Sales", alpha=0.35, s=18, ax=axes[0])
axes[0].set_title("Visibility vs Sales")
sns.scatterplot(data=df, x="Item Weight", y="Sales", alpha=0.35, s=18, ax=axes[1])
axes[1].set_title("Weight vs Sales")
sns.boxplot(data=df, x="Rating", y="Sales", ax=axes[2])
axes[2].set_title("Sales by Rating")
plt.tight_layout()
plt.show()


## Objective 4 — Help management make better business decisions
**Tasks 4.1–4.4:** identify stronger contributors, areas to investigate, and practical recommendations grounded in the observed data.


In [ ]:
# Concise, reproducible findings from the supplied data
total_sales = df["Sales"].sum()
category_sales = df.groupby("Item Type")["Sales"].sum().sort_values(ascending=False)
outlet_sales = df.groupby("Outlet Type")["Sales"].sum().sort_values(ascending=False)
tier_sales = df.groupby("Outlet Location Type")["Sales"].sum().sort_values(ascending=False)
size_sales = df.groupby("Outlet Size")["Sales"].sum().sort_values(ascending=False)
corr = df[["Item Visibility", "Item Weight", "Sales", "Rating"]].corr(numeric_only=True)["Sales"]

print(f"Total recorded sales: {total_sales:,.2f}")
print(f"Top item type: {category_sales.index[0]} ({category_sales.iloc[0]:,.2f})")
print(f"Top outlet type: {outlet_sales.index[0]} ({outlet_sales.iloc[0]:,.2f})")
print(f"Top location tier: {tier_sales.index[0]} ({tier_sales.iloc[0]:,.2f})")
print(f"Top outlet size by total: {size_sales.index[0]} ({size_sales.iloc[0]:,.2f})")
print("\nCorrelation with sales (descriptive only):")
display(corr.rename("Correlation with Sales").to_frame())


### Business recommendations
1. **Protect strong category contributors.** Fruits and Vegetables and Snack Foods contribute the highest category-level recorded sales in this dataset. Validate availability and shelf execution for these categories before reallocating inventory.
2. **Investigate outlet mix, not just totals.** Supermarket Type1 has the highest aggregate sales and the largest number of records. Compare like-for-like outlet performance and sales per record before concluding that format alone explains performance.
3. **Use location-tier totals with context.** Tier 3 contributes the largest total sales, but it also has the most records. Monitor average sales per record alongside totals.
4. **Improve data capture.** Item Weight is missing for a material share of rows. Preserve the raw data and record weight at source where possible; use imputed weights only with clear labels.
5. **Keep product labels standardized.** Combine `LF`/`low fat` with `Low Fat` and `reg` with `Regular` for consistent reporting.
6. **Do not infer causation from weak correlations.** Visibility, weight, and rating have very weak linear correlations with sales in this dataset. Consider additional variables and a more suitable study design before making causal claims.
7. **Request richer data for next-step analysis.** Transaction dates, quantities, costs/profit, stock availability, promotions, and unique transaction IDs would enable time trends, profitability, stockout analysis, and more reliable outlier investigation.


## Limitations
- `Sales` is treated as the recorded sales measure supplied in the workbook; no currency unit is explicitly specified in the source workbook.
- The data has no transaction date, so time-series sales growth cannot be analyzed. `Outlet Establishment Year` is not a transaction date.
- The data has no profit, discount, order status, payment, or customer identifier columns; those topics are outside the scope of this dataset.
- IQR flags are review prompts, not proof of incorrect data.
- Correlation does not establish causation.
- Imputed item weights are estimates; the original file is retained separately.


In [ ]:
# Export clean data and summary tables for reuse
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

df_clean.to_csv(PROJECT_DIR / "data" / "blinkit_grocery_cleaned.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "sales_by_item_type.csv", index=False)
outlet_type_summary.to_csv(OUTPUT_DIR / "sales_by_outlet_type.csv", index=False)
location_summary.to_csv(OUTPUT_DIR / "sales_by_location_tier.csv", index=False)
size_summary.to_csv(OUTPUT_DIR / "sales_by_outlet_size.csv", index=False)
missing_report.to_csv(OUTPUT_DIR / "data_quality_missing_values.csv")
outlier_report.to_csv(OUTPUT_DIR / "data_quality_outlier_flags.csv", index=False)

print("Exports saved to the data/ and outputs/ folders.")
